# VGG11: physical nonidealities

Model each physical effect separately, then measure its impact on **all 10,000
CIFAR-10 test images**. Each section explains the model, declares its parameter
values, and prints its own accuracy table. All other device effects are disabled
within that section. Digital inference and ideal-device CIM provide common
references.

Install MemFast, torchvision, NumPy and Jupyter. Set `MEMFAST_CIFAR10_ROOT` to
an extracted torchvision-compatible CIFAR-10 directory, and
`MEMFAST_VGG_CHECKPOINT` to `cifar10_vgg11_bn-eaeebf42.pt` from the
[pytorch-cifar-models VGG release](https://github.com/chenyaofo/pytorch-cifar-models/releases/tag/vgg).
The cells below show the evaluated configuration and retain their actual outputs.

In [1]:
import gc
import os
from dataclasses import asdict, replace
from pathlib import Path

import torch
from IPython.display import Markdown, display
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from memfast import Config, Nonidealities, Simulator, convert_model
from memfast.models import cifar_model

SEED = 135
torch.manual_seed(SEED)
torch.set_num_threads(4)
device = torch.device(
    os.environ.get("MEMFAST_NOTEBOOK_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu")
)
data_root = Path(os.environ.get("MEMFAST_CIFAR10_ROOT", "data/cifar10"))
checkpoint = Path(
    os.environ.get(
        "MEMFAST_VGG_CHECKPOINT",
        str(Path(torch.hub.get_dir()) / "checkpoints/cifar10_vgg11_bn-eaeebf42.pt"),
    )
)
batch_size = 8
print("Device:", device, "batch size:", batch_size, "seed:", SEED)

Device: cuda:0 batch size: 8 seed: 135


## 1. Load the checkpoint and full test set

The model definition is `cifar_model("vgg11_bn", num_classes=10)`.
Load the original digital weights strictly before converting layers. Every
Conv2d and Linear layer is mapped; normalization, activation, pooling and biases
remain digital. Evaluation uses the fixed test order without augmentation.

In [2]:
preprocess = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)
dataset = datasets.CIFAR10(data_root, train=False, download=False, transform=preprocess)
assert len(dataset) == 10000
batches = list(DataLoader(dataset, batch_size=batch_size, shuffle=False))
targets = torch.cat([labels for _, labels in batches])
model = cifar_model("vgg11_bn", num_classes=10).float().eval()
payload = torch.load(checkpoint, map_location="cpu", weights_only=True)
model.load_state_dict(payload.get("state_dict", payload), strict=True)
layer_names = [
    name for name, layer in model.named_modules() if isinstance(layer, (nn.Conv2d, nn.Linear))
]
print("Checkpoint:", checkpoint.name)
print("Test images:", len(targets), "mapped layers:", len(layer_names))
print("Layers:", ", ".join(layer_names))


@torch.inference_mode()
def evaluate(network):
    network.eval()
    predictions = []
    for images, _ in batches:
        predictions.append(network(images.to(device)).cpu())
    logits = torch.cat(predictions)
    assert logits.shape == (10000, 10) and torch.isfinite(logits).all()
    return {
        "count": len(targets),
        "correct": int((logits.argmax(1) == targets).sum()),
        "top1": float((logits.argmax(1) == targets).double().mean()),
        "cross_entropy": float(nn.functional.cross_entropy(logits, targets)),
        "logits": logits,
    }

Checkpoint: cifar10_vgg11_bn-eaeebf42.pt
Test images: 10000 mapped layers: 11
Layers: features.0, features.4, features.8, features.11, features.15, features.18, features.22, features.25, classifier.0, classifier.3, classifier.6


## 2. Fix the quantization and ADC configuration

All experiments use signed A8/W8 INT, one-bit slices, 64x64 arrays,
64 active rows per ADC read, ADC8, per-vector input scales and per-output-channel
weight scales. ADC8 covers the ideal binary-slice counts 0..64. Restoring input
and weight scales happens after each ADC read, before digital accumulation.
These settings stay fixed throughout the notebook.

For a weight-slice digit $q_w$ with maximum $m_w$ and input digit $q_a$ with
maximum $m_a$,

$$G_0=G_{\min}+(G_{\max}-G_{\min})q_w/m_w,\qquad
V_0=V_{\mathrm{read}}q_a/m_a.$$

The physical current is $\sum_k V_k(G_k-G_{\min})$.
ADC conversion acts on the normalized digit current
$u=m_a m_w I/I_{\mathrm{unit}}$, where
$I_{\mathrm{unit}}=(G_{\max}-G_{\min})V_{\mathrm{read}}$.
Signed slice significance and operand scales are restored digitally.

Ideal devices still include quantization and ADC conversion.
`Nonidealities(write_variation=0.0)` explicitly disables the package's default
write variation for the reference and for every independent sweep below.

In [3]:
ideal_effects = Nonidealities(write_variation=0.0)
base_config = Config(
    activation_bits=8,
    weight_bits=8,
    input_slice=(1,) * 8,
    weight_slice=(1,) * 8,
    array_size=(64, 64),
    rows_per_read=64,
    adc_bits=8,
    input_quant_gran=(1, None),
    weight_quant_gran=(None, 1),
    seed=SEED,
    nonidealities=ideal_effects,
)
print(asdict(base_config))
model.to(device)
digital_result = evaluate(model)
model.cpu()
evaluations = {}


def evaluate_config(config):
    if config not in evaluations:
        engine = Simulator(config, device=str(device))
        analog = convert_model(model, engine)
        result = evaluate(analog)
        result["agreement"] = float(
            (result["logits"].argmax(1) == digital_result["logits"].argmax(1)).double().mean()
        )
        evaluations[config] = result
        del analog, engine
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()
    return evaluations[config]


ideal_result = evaluate_config(base_config)
print(f"Digital: {digital_result['correct']}/10000 = {100 * digital_result['top1']:.2f}%")
print(f"Ideal CIM: {ideal_result['correct']}/10000 = {100 * ideal_result['top1']:.2f}%")

{'activation_bits': 8, 'weight_bits': 8, 'array_size': (64, 64), 'weight_quant_gran': (None, 1), 'input_quant_gran': (1, None), 'adc_bits': 8, 'high_conductance': 1e-05, 'low_conductance': 1e-07, 'read_voltage': 0.2, 'seed': 135, 'nonidealities': {'write_variation': 0.0, 'read_variation': 0.0, 'read_mode': 'shared', 'voltage_variation': 0.0, 'adc_nonlinearity': 0.0, 'stuck_at_low': 0.0, 'stuck_at_high': 0.0, 'drift_coefficient': 0.0, 'drift_time': 0.0, 'drift_reference_time': 1.0}, 'rows_per_read': 64, 'input_slice': (1, 1, 1, 1, 1, 1, 1, 1), 'weight_slice': (1, 1, 1, 1, 1, 1, 1, 1), 'bw_e': None, 'weight_signed': True}


Digital: 9278/10000 = 92.78%
Ideal CIM: 9279/10000 = 92.79%


Each table includes the zero-effect reference, correct predictions out of 10,000,
top-1 accuracy, change from ideal-device CIM in percentage points, and prediction
agreement with the digital model. Identical configurations reuse the same
evaluation. Edit the declared values in each section to study your own devices.

In [4]:
def scan(title, values, make_effects, parameter="Parameter"):
    lines = [
        f"### {title}",
        f"| {parameter} | Correct / 10,000 | Top-1 (%) | Change vs ideal (pp) | Digital agreement (%) |",
        "| --- | ---: | ---: | ---: | ---: |",
    ]
    rows = []
    for value in values:
        effects = make_effects(value)
        config = replace(base_config, nonidealities=effects)
        result = evaluate_config(config)
        rows.append((value, config, result))
        delta = 100 * (result["top1"] - ideal_result["top1"])
        lines.append(
            f"| {value} | {result['correct']} | {100 * result['top1']:.2f} "
            f"| {delta:+.2f} | {100 * result['agreement']:.2f} |"
        )
        print(f"{title}: {parameter}={value}, {result['correct']}/10000", flush=True)
    display(Markdown("\n".join(lines)))
    return rows

## 3. Write variation

$$G_{\mathrm{program}}=G_0\exp(\sigma_w z),\qquad z\sim\mathcal N(0,1).$$

`write_variation` is the standard deviation in log-conductance, not an additive
percentage of the final current. Random values are fixed per physical cell,
weight slice, matrix name and seed. Programming variation is shared across
all inputs and read events. There is no endpoint clipping after this variation.
The sweep changes only $\sigma_w$.

In [5]:
write_rows = scan(
    "Write variation",
    (0.0, 0.02, 0.05, 0.10, 0.20),
    lambda value: replace(ideal_effects, write_variation=value),
    "sigma_w",
)

Write variation: sigma_w=0.0, 9279/10000


Write variation: sigma_w=0.02, 9279/10000


Write variation: sigma_w=0.05, 9275/10000


Write variation: sigma_w=0.1, 9054/10000


Write variation: sigma_w=0.2, 6605/10000


### Write variation
| sigma_w | Correct / 10,000 | Top-1 (%) | Change vs ideal (pp) | Digital agreement (%) |
| --- | ---: | ---: | ---: | ---: |
| 0.0 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.02 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.05 | 9275 | 92.75 | -0.04 | 99.13 |
| 0.1 | 9054 | 90.54 | -2.25 | 94.54 |
| 0.2 | 6605 | 66.05 | -26.74 | 67.55 |

## 4. Read variation: shared cell field

$$G'=G_{\mathrm{program}}\exp(\sigma_r z_{\mathrm{cell,event}}).$$

`read_mode="shared"` samples a fresh cell field for each forward event.
The field is **shared by all vectors in the batch and all input slices**,
and independent between weight slices. A convolution vector is an unfolded
image patch. Matrix names, seed and event number determine the draws.
All tables keep batch size 8 and sample order fixed.

In [6]:
shared_rows = scan(
    "Shared-cell read variation",
    (0.0, 0.02, 0.05, 0.10),
    lambda value: replace(ideal_effects, read_variation=value, read_mode="shared"),
    "sigma_r",
)

Shared-cell read variation: sigma_r=0.0, 9279/10000


Shared-cell read variation: sigma_r=0.02, 9279/10000


Shared-cell read variation: sigma_r=0.05, 9269/10000


Shared-cell read variation: sigma_r=0.1, 8946/10000


### Shared-cell read variation
| sigma_r | Correct / 10,000 | Top-1 (%) | Change vs ideal (pp) | Digital agreement (%) |
| --- | ---: | ---: | ---: | ---: |
| 0.0 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.02 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.05 | 9269 | 92.69 | -0.10 | 99.19 |
| 0.1 | 8946 | 89.46 | -3.33 | 93.46 |

## 5. Read variation: independent input/weight slice pairs

`read_mode="slice_pair"` samples a separate cell field for each event and
input/weight slice pair. Vectors within the batch still share that field.
With four one-bit input slices and four one-bit weight slices there are
**16 independent fields**, rather than four fields reused across input slices.
The marginal model remains $G'=G\exp(\sigma_r z)$; only its correlations change.

In [7]:
pair_rows = scan(
    "Slice-pair read variation",
    (0.0, 0.02, 0.05, 0.10),
    lambda value: replace(ideal_effects, read_variation=value, read_mode="slice_pair"),
    "sigma_r",
)

Slice-pair read variation: sigma_r=0.0, 9279/10000


Slice-pair read variation: sigma_r=0.02, 9279/10000


Slice-pair read variation: sigma_r=0.05, 9275/10000


Slice-pair read variation: sigma_r=0.1, 8951/10000


### Slice-pair read variation
| sigma_r | Correct / 10,000 | Top-1 (%) | Change vs ideal (pp) | Digital agreement (%) |
| --- | ---: | ---: | ---: | ---: |
| 0.0 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.02 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.05 | 9275 | 92.75 | -0.04 | 98.99 |
| 0.1 | 8951 | 89.51 | -3.28 | 93.47 |

## 6. Read variation: output-current approximation

`read_mode="output"` approximates the same lognormal cell variation by a
Gaussian current perturbation. Conditional on the programmed $G_k$ and input
voltages $V_k$, the added current has moments

$$\mu=(e^{\sigma_r^2/2}-1)\sum_k V_kG_k,\qquad
\tau^2=e^{\sigma_r^2}(e^{\sigma_r^2}-1)\sum_k V_k^2G_k^2.$$

The perturbation is $\mu+\tau z$, before ADC conversion. Samples are independent
per input vector, physical subread, output column, slice pair and event.
This provides finer sampling than a batch-shared cell field. It is a
moment-matched approximation, with different correlations from the two modes
above. The nominal $G_{\min}$ floor is still subtracted from the current.

In [8]:
read_output_rows = scan(
    "Output-equivalent read variation",
    (0.0, 0.02, 0.05, 0.10),
    lambda value: replace(ideal_effects, read_variation=value, read_mode="output"),
    "sigma_r",
)

Output-equivalent read variation: sigma_r=0.0, 9279/10000


Output-equivalent read variation: sigma_r=0.02, 9279/10000


Output-equivalent read variation: sigma_r=0.05, 9267/10000


Output-equivalent read variation: sigma_r=0.1, 8700/10000


### Output-equivalent read variation
| sigma_r | Correct / 10,000 | Top-1 (%) | Change vs ideal (pp) | Digital agreement (%) |
| --- | ---: | ---: | ---: | ---: |
| 0.0 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.02 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.05 | 9267 | 92.67 | -0.12 | 98.96 |
| 0.1 | 8700 | 87.00 | -5.79 | 89.86 |

## 7. Input-voltage variation

$$V'=V_0(1+\sigma_v z).$$

`voltage_variation` applies a multiplicative Gaussian perturbation to each
input-row voltage. Sampling is per input vector, row array, input slice,
row and event. The voltage is shared by output columns and weight slices
driven by that row. It is not clipped after perturbation.

In [9]:
voltage_rows = scan(
    "Voltage variation",
    (0.0, 0.02, 0.05, 0.10),
    lambda value: replace(ideal_effects, voltage_variation=value),
    "sigma_v",
)

Voltage variation: sigma_v=0.0, 9279/10000


Voltage variation: sigma_v=0.02, 9279/10000


Voltage variation: sigma_v=0.05, 9267/10000


Voltage variation: sigma_v=0.1, 9010/10000


### Voltage variation
| sigma_v | Correct / 10,000 | Top-1 (%) | Change vs ideal (pp) | Digital agreement (%) |
| --- | ---: | ---: | ---: | ---: |
| 0.0 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.02 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.05 | 9267 | 92.67 | -0.12 | 99.29 |
| 0.1 | 9010 | 90.10 | -2.69 | 93.84 |

## 8. ADC nonlinearity

For normalized digit current $u$ and $M=2^{b_{\mathrm{ADC}}}-1$,
the implemented transfer is

$$Q(u)=\operatorname{round}\left[
\operatorname{clip}\left(\frac{u}{1+|\eta u/M|},0,M\right)\right].$$

`adc_nonlinearity` sets $\eta$. Zero gives the linear clipped ADC;
positive values compress larger currents before rounding. This is a
deterministic transfer applied separately to every slice-pair subread,
not an extra random field. The ADC width stays at 8 in this sweep.

In [10]:
adc_rows = scan(
    "ADC nonlinearity",
    (0.0, 0.02, 0.10, 0.50),
    lambda value: replace(ideal_effects, adc_nonlinearity=value),
    "eta",
)

ADC nonlinearity: eta=0.0, 9279/10000


ADC nonlinearity: eta=0.02, 9279/10000


ADC nonlinearity: eta=0.1, 9279/10000


ADC nonlinearity: eta=0.5, 9238/10000


### ADC nonlinearity
| eta | Correct / 10,000 | Top-1 (%) | Change vs ideal (pp) | Digital agreement (%) |
| --- | ---: | ---: | ---: | ---: |
| 0.0 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.02 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.1 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.5 | 9238 | 92.38 | -0.41 | 98.01 |

## 9. Stuck-at-low faults

A uniform sample $U$ is fixed per physical cell and weight slice.
If $U<p_{\mathrm{low}}$, the conductance is forced to $G_{\min}$.
The fault map is shared by all input vectors and read events. This section
changes `stuck_at_low` while `stuck_at_high=0`.

In [11]:
low_rows = scan(
    "Stuck-at-low faults",
    (0.0, 0.001, 0.01, 0.05),
    lambda value: replace(ideal_effects, stuck_at_low=value),
    "p_low",
)

Stuck-at-low faults: p_low=0.0, 9279/10000


Stuck-at-low faults: p_low=0.001, 9266/10000


Stuck-at-low faults: p_low=0.01, 9035/10000


Stuck-at-low faults: p_low=0.05, 6371/10000


### Stuck-at-low faults
| p_low | Correct / 10,000 | Top-1 (%) | Change vs ideal (pp) | Digital agreement (%) |
| --- | ---: | ---: | ---: | ---: |
| 0.0 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.001 | 9266 | 92.66 | -0.13 | 98.67 |
| 0.01 | 9035 | 90.35 | -2.44 | 94.20 |
| 0.05 | 6371 | 63.71 | -29.08 | 65.16 |

## 10. Stuck-at-high faults

If $p_{\mathrm{low}}\leq U<p_{\mathrm{low}}+p_{\mathrm{high}}$,
the conductance is forced to $G_{\max}$. Low and high faults are mutually
exclusive, so their probabilities must sum to at most one.
Fault assignment follows programming variation and retention drift, and
precedes event-dependent read noise. This section changes `stuck_at_high`
while `stuck_at_low=0`.

In [12]:
high_rows = scan(
    "Stuck-at-high faults",
    (0.0, 0.001, 0.01, 0.05),
    lambda value: replace(ideal_effects, stuck_at_high=value),
    "p_high",
)

Stuck-at-high faults: p_high=0.0, 9279/10000


Stuck-at-high faults: p_high=0.001, 9256/10000


Stuck-at-high faults: p_high=0.01, 8937/10000


Stuck-at-high faults: p_high=0.05, 6063/10000


### Stuck-at-high faults
| p_high | Correct / 10,000 | Top-1 (%) | Change vs ideal (pp) | Digital agreement (%) |
| --- | ---: | ---: | ---: | ---: |
| 0.0 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.001 | 9256 | 92.56 | -0.23 | 98.47 |
| 0.01 | 8937 | 89.37 | -3.42 | 93.23 |
| 0.05 | 6063 | 60.63 | -32.16 | 61.65 |

## 11. Retention drift

$$G(t)=G_{\mathrm{program}}(1+t/t_{\mathrm{ref}})^{-\nu}.$$

`drift_coefficient` is $\nu$; `drift_time` and `drift_reference_time` are in
seconds. The configured exponent is common to the cells; this model does not
sample a distribution of cell exponents. The first table varies $\nu$ at
$t=1000$ s and $t_{\mathrm{ref}}=1$ s. The second varies retention time at
$\nu=0.03$, using the same checkpoint and quantization throughout.

In [13]:
drift_rows = scan(
    "Drift exponent at t=1000 s",
    (0.0, 0.01, 0.03, 0.06),
    lambda value: replace(
        ideal_effects, drift_coefficient=value, drift_time=1000.0, drift_reference_time=1.0
    ),
    "nu",
)
drift_time_rows = scan(
    "Retention time at nu=0.03",
    (0.0, 1.0, 10.0, 100.0, 1000.0),
    lambda value: replace(
        ideal_effects, drift_coefficient=0.03, drift_time=value, drift_reference_time=1.0
    ),
    "Time (s)",
)

Drift exponent at t=1000 s: nu=0.0, 9279/10000


Drift exponent at t=1000 s: nu=0.01, 9099/10000


Drift exponent at t=1000 s: nu=0.03, 6555/10000


Drift exponent at t=1000 s: nu=0.06, 2211/10000


### Drift exponent at t=1000 s
| nu | Correct / 10,000 | Top-1 (%) | Change vs ideal (pp) | Digital agreement (%) |
| --- | ---: | ---: | ---: | ---: |
| 0.0 | 9279 | 92.79 | +0.00 | 99.75 |
| 0.01 | 9099 | 90.99 | -1.80 | 95.00 |
| 0.03 | 6555 | 65.55 | -27.24 | 66.78 |
| 0.06 | 2211 | 22.11 | -70.68 | 22.75 |

...

Retention time at nu=0.03: Time (s)=1.0, 9281/10000


Retention time at nu=0.03: Time (s)=10.0, 9069/10000


Retention time at nu=0.03: Time (s)=100.0, 8176/10000


Retention time at nu=0.03: Time (s)=1000.0, 6555/10000


### Retention time at nu=0.03
| Time (s) | Correct / 10,000 | Top-1 (%) | Change vs ideal (pp) | Digital agreement (%) |
| --- | ---: | ---: | ---: | ---: |
| 0.0 | 9279 | 92.79 | +0.00 | 99.75 |
| 1.0 | 9281 | 92.81 | +0.02 | 99.79 |
| 10.0 | 9069 | 90.69 | -2.10 | 94.51 |
| 100.0 | 8176 | 81.76 | -11.03 | 83.78 |
| 1000.0 | 6555 | 65.55 | -27.24 | 66.78 |

## 12. Compose a device configuration

Once you have inspected the individual effects, combine the fields for your
target device. The object below is an editable example; all measured tables
above vary one effect at a time. Use this configuration in the same public
`Simulator` and `convert_model` calls to evaluate a combined model.

In [14]:
custom_config = replace(
    base_config,
    nonidealities=Nonidealities(
        write_variation=0.02,
        read_variation=0.02,
        read_mode="slice_pair",
    ),
)
print(asdict(custom_config))
assert all(result["count"] == 10000 for result in evaluations.values())
print("Completed independent CIM configurations:", len(evaluations))
print("Each configuration evaluated all 10,000 CIFAR-10 test images.")

{'activation_bits': 8, 'weight_bits': 8, 'array_size': (64, 64), 'weight_quant_gran': (None, 1), 'input_quant_gran': (1, None), 'adc_bits': 8, 'high_conductance': 1e-05, 'low_conductance': 1e-07, 'read_voltage': 0.2, 'seed': 135, 'nonidealities': {'write_variation': 0.02, 'read_variation': 0.02, 'read_mode': 'slice_pair', 'voltage_variation': 0.0, 'adc_nonlinearity': 0.0, 'stuck_at_low': 0.0, 'stuck_at_high': 0.0, 'drift_coefficient': 0.0, 'drift_time': 0.0, 'drift_reference_time': 1.0}, 'rows_per_read': 64, 'input_slice': (1, 1, 1, 1, 1, 1, 1, 1), 'weight_slice': (1, 1, 1, 1, 1, 1, 1, 1), 'bw_e': None, 'weight_signed': True}
Completed independent CIM configurations: 36
Each configuration evaluated all 10,000 CIFAR-10 test images.
